# DeepSentinel -- TS-TCN Ablation Study (Stage 5)

**Project:** R26-IT-121 * **Author:** Pathirana P.K.V. (IT22237972) * **Member 3**

Trains ONE ablation arm (A2, A3, or A4 -- **A1 is the primary run, already covered
by `DeepSentinel_T4_FullTraining.ipynb`, do not redo it here**) per proposal Table 3.4:

| ID | Config | Change from primary |
|----|--------|----------------------|
| A1 | W=32 + attention (primary) | -- (run separately) |
| A2 | W=32, no attention | Remove fraud_attention's contribution to the head |
| A3 | W=16 + attention | New W=16 window tensors |
| A4 | W=64 + attention | New W=64 window tensors |

## Running this in parallel (recommended)

Open **one Colab tab per arm** (A2, A3, A4 -- 3 tabs, plus the primary notebook's
tab for A1 = 4 tabs total), each on its own GPU runtime. In **Cell 1 (CONFIG)**
of each tab, set `ABLATION_ID` to `"A2"`, `"A3"`, or `"A4"` -- everything else
adapts automatically. Run all cells top to bottom in each tab. They do not
interact with each other, so running them at the same time is safe.

A2 needs no new data (reuses the existing W=32 tfrecords) -- it is the fastest
arm and a good one to start first as a sanity check that this notebook works.

A3/A4 build brand-new window tensors before training (Cell 5) -- this is a
single-threaded, CPU-bound pass over the full 6.36M-row feature table and gets
**no benefit from the GPU**, so it is safe (and faster overall) to run that cell
on a CPU-only Colab runtime, then switch the runtime to GPU (Runtime > Change
runtime type) before Cell 11 (training).

## Once all 4 arms have finished

Run the **last two cells** of this notebook (in any one tab) to merge
`A1..A4` results into `ablation_results.json` and produce the ablation
results chart (proposal Figure A4). Do this last, after every arm's own
summary cell has completed.

## Produces (per arm, in its own Drive output folder)
- `{ABLATION_ID}_model.keras`
- `{ABLATION_ID}_training_history.csv`
- `{ABLATION_ID}_metrics.json`
- `{ABLATION_ID}_training_curves.png`

## Produces (merge step, once)
- `ablation_results.json`
- `ablation_results_chart.png` (Figure A4)

## Cell 0 -- Environment Setup

In [ ]:
# =================================================================================
# Cell 0 -- Environment Setup (identical pattern to the primary training notebook)
# =================================================================================
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, gc, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.metrics import (precision_recall_curve, f1_score, precision_score,
                             recall_score, roc_auc_score, confusion_matrix)

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "#F9F9F9",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.grid": True, "grid.color": "#E0E0E0", "grid.linestyle": "--", "grid.alpha": 0.6,
})
DS_BLUE="#1A5276"; DS_RED="#C0392B"; DS_GREEN="#1E8449"; DS_ORANGE="#D35400"

DRIVE_BASE = Path("/content/drive/MyDrive/DeepSentinel")
T2_DIR    = DRIVE_BASE / "Member3_Stage3" / "outputs_t2"     # existing W=32 tfrecords
BASE_DIR  = DRIVE_BASE / "Member3_Baseline" / "outputs"      # scaler.pkl, baseline_metrics.json, features.parquet
T4_DIR    = DRIVE_BASE / "Member3_Stage4" / "outputs_t4"     # primary run's outputs (for A1's numbers later)
ABL_ROOT  = DRIVE_BASE / "Member3_Stage5_Ablation"
ABL_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print("Environment ready")
print(f"   TensorFlow : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"   GPU        : {gpus or 'NONE -- CPU only (fine for window-building, NOT for training)'}")

## Cell 1 -- CONFIG (the one cell to change per tab)

Set `ABLATION_ID` here and re-run the whole notebook. Everything downstream
(window size, attention on/off, output paths) follows from this one value.

In [ ]:
# =================================================================================
# Cell 1 -- CONFIG -- CHANGE ABLATION_ID PER TAB, NOTHING ELSE
# =================================================================================
ABLATION_ID = "A2"   # <-- one of "A2", "A3", "A4" -- set differently in each Colab tab

_CONFIGS = {
    "A2": {"window_size": 32, "use_attention": False, "desc": "W=32, no attention"},
    "A3": {"window_size": 16, "use_attention": True,  "desc": "W=16 + attention"},
    "A4": {"window_size": 64, "use_attention": True,  "desc": "W=64 + attention"},
}
assert ABLATION_ID in _CONFIGS, f"ABLATION_ID must be one of {list(_CONFIGS)}"
cfg = _CONFIGS[ABLATION_ID]
W = cfg["window_size"]
USE_ATTENTION = cfg["use_attention"]
F = 10

OUT_DIR = ABL_ROOT / ABLATION_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR = Path(f"/content/local_data_{ABLATION_ID}"); LOCAL_DIR.mkdir(exist_ok=True)

print(f"Ablation arm : {ABLATION_ID} -- {cfg['desc']}")
print(f"Window size  : W={W}")
print(f"Attention    : {'ON' if USE_ATTENTION else 'OFF (GlobalAvgPool1D feeds the head directly)'}")
print(f"Output dir   : {OUT_DIR}")

## Cell 2 -- FEATURE_NAMES + split config

Must match Stage 1/2/3 exactly -- these are the same 10 columns, same order,
same `split_step`, used everywhere else in the project.

In [ ]:
# =================================================================================
# Cell 2 -- Shared constants
# =================================================================================
FEATURE_NAMES = ["drain_ratio", "log_amount", "post_transfer_ratio", "dest_was_empty",
                  "dest_enrichment", "type_risk_weight", "inv_dest_ratio", "amt_to_orig",
                  "hour_of_day", "day_of_week"]
SPLIT_STEP = 595
BATCH_SIZE = 256
SHUFFLE_BUF = 50_000
AUTOTUNE = tf.data.AUTOTUNE

def _bytes_feature(v):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[v]))

def _int64_feature(v):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[v]))

def serialize_example(window, label, composite_id, step):
    feat = {
        "window": _bytes_feature(window.astype(np.float32).tobytes()),
        "label": _int64_feature(int(label)),
        "composite_id": _bytes_feature(str(composite_id).encode("utf-8")),
        "step": _int64_feature(int(step)),
    }
    return tf.train.Example(features=tf.train.Features(feature=feat)).SerializeToString()

## Cell 3 -- Load scaled features (once, from Stage 1/2)

Loads `features.parquet` (Stage 1) and `scaler.pkl` (Stage 2, fit on the
training partition only -- same scaler every arm uses, so results stay
comparable). If `features.parquet` was saved somewhere other than `BASE_DIR`
on your Drive, edit `FEATURES_PATH` below before running.

In [ ]:
# =================================================================================
# Cell 3 -- Load features.parquet + scaler.pkl (shared across all ablation arms)
# =================================================================================
import joblib

FEATURES_PATH = BASE_DIR / "features.parquet"   # <-- edit if your Drive layout differs
SCALER_PATH = BASE_DIR / "scaler.pkl"

assert FEATURES_PATH.exists(), (
    f"{FEATURES_PATH} not found. This must be the Stage 1 output with columns "
    f"{FEATURE_NAMES} + ['step','isFraud','nameOrig']. Point FEATURES_PATH at wherever "
    f"features.parquet actually lives on your Drive and re-run this cell."
)
assert SCALER_PATH.exists(), f"{SCALER_PATH} not found -- same scaler.pkl used elsewhere in the project."

print("Loading features.parquet ...")
t0 = time.time()
df = pd.read_parquet(FEATURES_PATH)
df = df.sort_values(["step"]).reset_index(drop=True)   # chronological order, matches Stage 3
scaler = joblib.load(SCALER_PATH)
print(f"   {len(df):,} rows loaded in {time.time()-t0:.0f}s")
assert all(c in df.columns for c in FEATURE_NAMES), "features.parquet is missing expected feature columns"
assert "isFraud" in df.columns and "step" in df.columns and "nameOrig" in df.columns

## Cell 4 -- Verify inputs / build windows at this arm's W

If `W == 32`, reuses the existing Stage 3 tfrecords (no rebuild, matches the
primary run's data exactly). Otherwise streams the full sorted feature table
through a `deque(maxlen=W)` -- the same system-wide sliding-window construction
as Stage 3 (`src/data/window_builder.py`), just at a different W -- and writes
new train/test tfrecords, cached under this arm's output folder so re-running
the notebook later does not rebuild them again.

**CPU-bound, no GPU needed.** Safe to run on a CPU-only runtime.

In [ ]:
# =================================================================================
# Cell 4 -- Verify inputs / build windows at W (Novelty N1, per-arm)
# =================================================================================
LOCAL_TRAIN = LOCAL_DIR / "train_windows.tfrecord"
LOCAL_TEST  = LOCAL_DIR / "test_windows.tfrecord"
DRIVE_TRAIN = OUT_DIR / f"train_windows_w{W}.tfrecord"
DRIVE_TEST  = OUT_DIR / f"test_windows_w{W}.tfrecord"
META_PATH   = OUT_DIR / f"windows_metadata_w{W}.json"

if W == 32 and T2_DIR.exists():
    print("W=32 -- reusing the existing Stage 3 tfrecords (identical data to the primary run).")
    SRC_TRAIN = T2_DIR / "train_windows.tfrecord"
    SRC_TEST  = T2_DIR / "test_windows.tfrecord"
    assert SRC_TRAIN.exists() and SRC_TEST.exists(), f"Expected Stage 3 tfrecords under {T2_DIR}"
    with open(T2_DIR / "windows_metadata.json") as f:
        meta = json.load(f)
    N_TRAIN, N_TEST = meta["counts"]["train_windows"], meta["counts"]["test_windows"]
    shutil.copy(SRC_TRAIN, LOCAL_TRAIN); shutil.copy(SRC_TEST, LOCAL_TEST)

elif DRIVE_TRAIN.exists() and DRIVE_TEST.exists() and META_PATH.exists():
    print(f"W={W} windows already built for this arm -- reusing cached tfrecords from a previous run.")
    with open(META_PATH) as f:
        meta = json.load(f)
    N_TRAIN, N_TEST = meta["train_windows"], meta["test_windows"]
    shutil.copy(DRIVE_TRAIN, LOCAL_TRAIN); shutil.copy(DRIVE_TEST, LOCAL_TEST)

else:
    print(f"Building new W={W} window tensors from features.parquet (single pass, {len(df):,} rows) ...")
    print("This is CPU-bound and does not need a GPU.")
    from collections import deque

    features_arr = scaler.transform(df[FEATURE_NAMES].values.astype(np.float32))
    labels_arr = df["isFraud"].values.astype(np.int64)
    steps_arr = df["step"].values.astype(np.int64)
    cids_arr = (df["nameOrig"].astype(str) + "_" + df["step"].astype(str)).values

    buffer = deque(maxlen=W)
    counts = {"train_windows": 0, "test_windows": 0, "cold_start_skipped": 0,
              "fraud_train_windows": 0, "fraud_test_windows": 0}

    t0 = time.time()
    with tf.io.TFRecordWriter(str(LOCAL_TRAIN)) as tw, tf.io.TFRecordWriter(str(LOCAL_TEST)) as tsw:
        for i in range(len(df)):
            if len(buffer) == W:
                example = serialize_example(np.stack(buffer, axis=0), labels_arr[i], cids_arr[i], steps_arr[i])
                if steps_arr[i] <= SPLIT_STEP:
                    tw.write(example); counts["train_windows"] += 1
                    counts["fraud_train_windows"] += int(labels_arr[i])
                else:
                    tsw.write(example); counts["test_windows"] += 1
                    counts["fraud_test_windows"] += int(labels_arr[i])
            else:
                counts["cold_start_skipped"] += 1
            buffer.append(features_arr[i])
            if i % 1_000_000 == 0 and i > 0:
                print(f"   {i:,}/{len(df):,} rows ({time.time()-t0:.0f}s elapsed)")

    print(f"Window build finished in {(time.time()-t0)/60:.1f} min")
    N_TRAIN, N_TEST = counts["train_windows"], counts["test_windows"]
    print(f"   train={N_TRAIN:,} (fraud={counts['fraud_train_windows']})  "
          f"test={N_TEST:,} (fraud={counts['fraud_test_windows']})")

    # Cache to Drive so a notebook restart does not repeat this pass.
    shutil.copy(LOCAL_TRAIN, DRIVE_TRAIN); shutil.copy(LOCAL_TEST, DRIVE_TEST)
    with open(META_PATH, "w") as f:
        json.dump(counts, f, indent=2)
    print(f"Cached to {OUT_DIR}")

print(f"\nReady: W={W}, F={F}, train={N_TRAIN:,}, test={N_TEST:,}")

## Cell 5 -- Data Pipeline

In [ ]:
# =================================================================================
# Cell 5 -- tf.data pipeline (train repeats across epochs -- the Day-2/Day-3 fix)
# =================================================================================
FEATURE_DESCRIPTION = {
    "window": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
    "composite_id": tf.io.FixedLenFeature([], tf.string),
    "step": tf.io.FixedLenFeature([], tf.int64),
}
def parse_example(serialized):
    parsed = tf.io.parse_single_example(serialized, FEATURE_DESCRIPTION)
    window = tf.reshape(tf.io.decode_raw(parsed["window"], tf.float32), (W, F))
    label = tf.cast(parsed["label"], tf.float32)
    return window, label

def make_dataset(path, training=True):
    ds = tf.data.TFRecordDataset(str(path), num_parallel_reads=AUTOTUNE)
    ds = ds.map(parse_example, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(SHUFFLE_BUF, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.repeat()
    ds = ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(LOCAL_TRAIN, training=True)
test_ds  = make_dataset(LOCAL_TEST,  training=False)
STEPS_PER_EPOCH = N_TRAIN // BATCH_SIZE
TEST_STEPS = N_TEST // BATCH_SIZE + 1
print(f"Pipelines ready -- STEPS_PER_EPOCH={STEPS_PER_EPOCH:,}")

## Cell 6 -- Build TS-TCN for this arm

Same 4-block dilated causal TCN every arm. `USE_ATTENTION` (from Cell 1)
controls whether `fraud_attention`'s context vector feeds the dense head
(A1/A3/A4) or is bypassed in favour of `GlobalAvgPool1D` alone (A2, per
proposal Table 3.4). The model keeps two outputs either way -- when
attention is off, the second output is a constant zero vector with loss
weight 0 -- so every arm shares one training/evaluation pipeline.

In [ ]:
# =================================================================================
# Cell 6 -- Build TS-TCN (attention path controlled by USE_ATTENTION)
# =================================================================================
def dilated_causal_block(x, filters, dilation, dropout=0.2, block_name="block"):
    in_channels = x.shape[-1]; skip = x
    x = layers.Conv1D(filters, 3, padding="causal", dilation_rate=dilation,
                      kernel_initializer="he_normal", name=f"{block_name}_conv1")(x)
    x = layers.BatchNormalization(name=f"{block_name}_bn1")(x)
    x = layers.ReLU(name=f"{block_name}_relu1")(x)
    x = layers.Dropout(dropout, name=f"{block_name}_drop1")(x)
    x = layers.Conv1D(filters, 3, padding="causal", dilation_rate=dilation,
                      kernel_initializer="he_normal", name=f"{block_name}_conv2")(x)
    x = layers.BatchNormalization(name=f"{block_name}_bn2")(x)
    x = layers.ReLU(name=f"{block_name}_relu2")(x)
    x = layers.Dropout(dropout, name=f"{block_name}_drop2")(x)
    if in_channels != filters:
        skip = layers.Conv1D(filters, 1, padding="same", name=f"{block_name}_residual_proj")(skip)
    return layers.Add(name=f"{block_name}_add")([skip, x])

@keras.utils.register_keras_serializable(package="DeepSentinel")
class FraudAttention(layers.Layer):
    def __init__(self, d_k=32, **kwargs):
        super().__init__(**kwargs); self.d_k = d_k
    def build(self, input_shape):
        self.q_dense = layers.Dense(self.d_k, name="q_proj")
        self.k_dense = layers.Dense(self.d_k, name="k_proj")
        self.v_dense = layers.Dense(self.d_k, name="v_proj")
        self.scale = tf.cast(tf.math.sqrt(tf.cast(self.d_k, tf.float32)), tf.float32)
        super().build(input_shape)
    def call(self, x):
        Q=self.q_dense(x); K=self.k_dense(x); V=self.v_dense(x)
        scores = tf.matmul(Q, K, transpose_b=True) / self.scale
        weights = tf.nn.softmax(scores, axis=-1)
        centre_weights = weights[:, -1, :]
        context = tf.squeeze(tf.matmul(tf.expand_dims(centre_weights, 1), V), axis=1)
        return context, centre_weights
    def get_config(self):
        cfg = super().get_config(); cfg["d_k"] = self.d_k; return cfg

keras.backend.clear_session(); tf.random.set_seed(SEED)
DILATIONS=[1,2,4,8]; FILTERS=96; DROPOUT=0.2; ATTN_D_K=32

def build_ts_tcn_ablation(W, F, use_attention=True):
    inputs = keras.Input(shape=(W, F), name="window")
    x = inputs
    for i, d in enumerate(DILATIONS):
        x = dilated_causal_block(x, FILTERS, d, DROPOUT, f"tcn{i+1}_d{d}")
    pooled = layers.GlobalAveragePooling1D(name="global_pool")(x)
    if use_attention:
        context, attn = FraudAttention(d_k=ATTN_D_K, name="fraud_attention")(x)
        head_input = layers.Concatenate(name="concat_attn_pool")([context, pooled])
    else:
        head_input = pooled
        attn = layers.Lambda(lambda t: tf.zeros((tf.shape(t)[0], W)),
                             name="fraud_attention")(pooled)
    head = layers.Dense(64, activation="relu", name="head_dense")(head_input)
    head = layers.Dropout(0.3, name="head_drop")(head)
    fraud_prob = layers.Dense(1, activation="sigmoid", name="fraud_prob")(head)
    return Model(inputs, [fraud_prob, attn], name=f"TS_TCN_{ABLATION_ID}")

def zero_loss(y_true, y_pred):
    return tf.zeros_like(tf.reduce_mean(y_pred, axis=-1))

model = build_ts_tcn_ablation(W, F, use_attention=USE_ATTENTION)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={"fraud_prob": keras.losses.BinaryFocalCrossentropy(gamma=2.0), "fraud_attention": zero_loss},
    loss_weights={"fraud_prob": 1.0, "fraud_attention": 0.0},
    metrics={"fraud_prob": [keras.metrics.Precision(name="precision"),
                            keras.metrics.Recall(name="recall"),
                            keras.metrics.AUC(name="auc")]},
)
print(f"Model rebuilt for {ABLATION_ID} ({cfg['desc']}) -- {model.count_params():,} parameters")

## Cell 7 -- Train this ablation arm

Same callback budget as the primary run (30 epochs, EarlyStopping on
`val_fraud_prob_recall`, patience 5) so the comparison in Cell 8/Figure A4
is apples-to-apples with A1.

In [ ]:
# =================================================================================
# Cell 7 -- Train (identical callback budget to the primary A1 run)
# =================================================================================
def to_multi_output(window, label):
    return window, {"fraud_prob": label, "fraud_attention": tf.zeros([W], dtype=tf.float32)}

train_ds_mo = train_ds.map(to_multi_output, num_parallel_calls=AUTOTUNE)
test_ds_mo  = test_ds.map(to_multi_output,  num_parallel_calls=AUTOTUNE)

gpus = tf.config.list_physical_devices('GPU')
assert gpus, "No GPU visible -- switch this Colab runtime to GPU (Runtime > Change runtime type) before training."

BEST_MODEL_PATH = OUT_DIR / f"{ABLATION_ID}_model.keras"
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_fraud_prob_recall", mode="max",
        patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor="val_fraud_prob_recall",
        mode="max", save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
        patience=3, min_lr=1e-6, verbose=1),
]

EPOCHS = 30
print(f"Training {ABLATION_ID} ({cfg['desc']}) -- up to {EPOCHS} epochs ...\n")
t0 = time.time()
history = model.fit(train_ds_mo, validation_data=test_ds_mo, epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH, validation_steps=TEST_STEPS,
    callbacks=callbacks, verbose=1)
elapsed = time.time() - t0
print(f"\nTraining finished in {elapsed/60:.1f} min ({len(history.history['loss'])} epochs ran)")

pd.DataFrame(history.history).to_csv(OUT_DIR / f"{ABLATION_ID}_training_history.csv", index=False)

h = history.history; ep = range(1, len(h["loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"{ABLATION_ID} ({cfg['desc']}) -- Learning Curves", fontsize=14, fontweight="bold", color=DS_BLUE)
for ax, (key, vkey, title) in zip(axes, [
    ("loss","val_loss","Loss"), ("fraud_prob_recall","val_fraud_prob_recall","Recall"),
    ("fraud_prob_auc","val_fraud_prob_auc","AUC")]):
    ax.plot(ep, h[key], color=DS_BLUE, label="train", marker="o", ms=4)
    ax.plot(ep, h[vkey], color=DS_RED, label="val", marker="s", ms=4)
    ax.set_xlabel("Epoch"); ax.set_title(title, fontweight="bold"); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / f"{ABLATION_ID}_training_curves.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved model + history + curves -> {OUT_DIR}")

## Cell 8 -- Threshold Tuning + Final Metrics

In [ ]:
# =================================================================================
# Cell 8 -- Threshold tuning + final test metrics for this arm
# =================================================================================
print("Collecting test-set predictions ...")
y_true_list, y_prob_list = [], []
for xb, yb in test_ds:
    prob, _ = model.predict(xb, verbose=0)
    y_prob_list.append(prob.ravel()); y_true_list.append(yb.numpy().ravel())
y_true = np.concatenate(y_true_list); y_prob = np.concatenate(y_prob_list)
print(f"   {len(y_true):,} predictions ({int(y_true.sum())} fraud)")

prec, rec, thresholds = precision_recall_curve(y_true, y_prob)
f1s = 2*prec[:-1]*rec[:-1] / (prec[:-1]+rec[:-1]+1e-9)
best_idx = int(np.argmax(f1s)); best_threshold = float(thresholds[best_idx])
y_pred = (y_prob >= best_threshold).astype(int)

arm_f1 = f1_score(y_true, y_pred); arm_precision = precision_score(y_true, y_pred)
arm_recall = recall_score(y_true, y_pred); arm_auc = roc_auc_score(y_true, y_prob)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("-"*70)
print(f"  {ABLATION_ID} FINAL TEST METRICS ({cfg['desc']})")
print("-"*70)
print(f"   Threshold={best_threshold:.4f}  F1={arm_f1:.4f}  Precision={arm_precision:.4f}")
print(f"   Recall={arm_recall:.4f}  AUC={arm_auc:.4f}")
print(f"   TP={tp} FP={fp} FN={fn} TN={tn}   Fraud caught: {tp}/{tp+fn}")

arm_metrics = {
    "ablation_id": ABLATION_ID, "config": cfg["desc"], "window_size": W, "use_attention": USE_ATTENTION,
    "threshold": best_threshold, "f1": float(arm_f1), "precision": float(arm_precision),
    "recall": float(arm_recall), "auc": float(arm_auc),
    "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    "epochs_ran": len(history.history["loss"]), "elapsed_min": round(elapsed/60, 1),
}
with open(OUT_DIR / f"{ABLATION_ID}_metrics.json", "w") as f:
    json.dump(arm_metrics, f, indent=2)
print(f"\nSaved: {ABLATION_ID}_metrics.json")
print(f"\n{ABLATION_ID} DONE. If other arms are still running in their own tabs, leave them be --")
print("run the merge cells at the bottom of this notebook only once ALL arms have finished.")

---
## Merge step -- run ONLY after A1 (primary notebook), A2, A3 and A4 have all finished

Run these two cells in **any one tab**, once. They read every arm's saved
`*_metrics.json` (including A1's `tstcn_test_metrics.json` from the primary
notebook's `T4_DIR`) and produce the combined `ablation_results.json` plus
the grouped bar chart (proposal Figure A4).

In [ ]:
# =================================================================================
# Merge Cell 1 -- Collect A1-A4 results into ablation_results.json
# =================================================================================
combined = []

a1_path = T4_DIR / "tstcn_test_metrics.json"
if a1_path.exists():
    with open(a1_path) as f:
        a1 = json.load(f)
    combined.append({"ablation_id": "A1", "config": "W=32 + attention (primary)", "window_size": 32,
        "use_attention": True, "threshold": a1["threshold"], "f1": a1["f1"], "precision": a1["precision"],
        "recall": a1["recall"], "auc": a1["auc"]})
else:
    print(f"WARNING: {a1_path} not found -- run the primary notebook first, A1 will be missing from the chart.")

for arm_id in ["A2", "A3", "A4"]:
    p = ABL_ROOT / arm_id / f"{arm_id}_metrics.json"
    if p.exists():
        with open(p) as f:
            m = json.load(f)
        combined.append({k: m[k] for k in ["ablation_id","config","window_size","use_attention",
                                            "threshold","f1","precision","recall","auc"]})
    else:
        print(f"WARNING: {p} not found -- {arm_id} has not finished yet (or ran with a different OUT_DIR).")

assert combined, "No ablation results found at all -- nothing to merge."
results_path = ABL_ROOT / "ablation_results.json"
with open(results_path, "w") as f:
    json.dump(combined, f, indent=2)

comparison_df = pd.DataFrame(combined)
print(f"Saved: {results_path}\n")
print(comparison_df.to_string(index=False))

In [ ]:
# =================================================================================
# Merge Cell 2 -- Ablation results chart (Figure A4)
# =================================================================================
metrics_to_plot = ["f1", "recall", "auc"]
ids = comparison_df["ablation_id"].tolist()
x = np.arange(len(ids)); width = 0.25
colors = [DS_BLUE, DS_RED, DS_GREEN]

fig, ax = plt.subplots(figsize=(10, 6))
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i*width, comparison_df[metric], width, label=metric.upper(), color=colors[i], alpha=0.85)
labels = [f"{row.ablation_id}\n{row.config}" for row in comparison_df.itertuples()]
ax.set_xticks(x + width); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.set_title("Ablation Study Results -- Window Size and Attention Layer Impact",
             fontsize=13, fontweight="bold", color=DS_BLUE)
ax.legend(loc="upper right")
for i, metric in enumerate(metrics_to_plot):
    for j, v in enumerate(comparison_df[metric]):
        ax.text(x[j] + i*width, v + 0.01, f"{v:.3f}", ha="center", fontsize=7)
plt.tight_layout()
plt.savefig(ABL_ROOT / "ablation_results_chart.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {ABL_ROOT / 'ablation_results_chart.png'}")

if "A1" in ids and "A3" in ids:
    a1r, a3r = comparison_df.set_index("ablation_id").loc[["A1","A3"], "recall"]
    print(f"\nA1 vs A3 recall: {a1r:.4f} vs {a3r:.4f} -- "
          f"{'confirms W=32 needed' if a1r > a3r else 'A3 did NOT come in lower than A1 -- re-check before writing this up as confirming the hypothesis'}")
if "A1" in ids and "A2" in ids:
    a1r, a2r = comparison_df.set_index("ablation_id").loc[["A1","A2"], "f1"]
    print(f"A1 vs A2 F1: {a1r:.4f} vs {a2r:.4f} -- "
          f"{'confirms attention adds signal' if a1r > a2r else 'A2 did NOT come in lower than A1 -- re-check before writing this up as confirming the hypothesis'}")